In [ ]:
import sys
sys.path.append('E:\machine learn\KAN\pykan-master\pykan-master')
import importlib
import kan
from kan import *
importlib.reload(kan)
import torch
import torch.optim as optim
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
import copy
# import shap
# import seaborn as sns
import time
# import pykan

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(torch.version.cuda)
# device = 'cpu'
print(device)

torch.set_default_dtype(torch.float64)

None
cpu


**导入数据**

数据归一化预处理

In [2]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
# 设置随机种子以确保结果可复现
# 读取CSV文件
# print('ts')
# data = pd.read_csv(r'E:\machine learn\KAN\pykan-master\pykan-master\data\data_parameter.csv')
data = pd.read_csv(r'E:\machine learn\KAN\pykan-master\pykan-master\data\diadata.csv')
#print(data.head())

# 选择输入特征和输出标签
input_cols = data.columns[0:12]
output_cols = data.columns[12:16]
#print("输入特征列:", input_cols)


# 转换为numpy数组并转为torch.Tensor
xdata = data[input_cols].values.astype(np.float64)
ydata = data[output_cols].values.astype(np.float64)

# # 归一化只用训练集fit
scaler_xdata = MinMaxScaler()
scaler_ydata = MinMaxScaler()

xdata_n = scaler_xdata.fit_transform(xdata)
ydata_n = scaler_ydata.fit_transform(ydata)
# print(xdata_n,ydata_n)


xdata = torch.tensor(xdata_n, dtype=torch.float64, device=device)
ydata = torch.tensor(ydata_n, dtype=torch.float64, device=device)



# # 按8:2划分训练集和测试集，random_state保证可复现
# x_train, x_test, y_train, y_test = train_test_split(
#     x, y, test_size=0.2, shuffle=True, random_state=42
# )
# # print(x_train)
# # print(x_train[1:5])
# # # 归一化只用训练集fit
# scaler_x = MinMaxScaler()
# scaler_y = MinMaxScaler()
# x_train_norm = scaler_x.fit_transform(x_train)
# y_train_norm = scaler_y.fit_transform(y_train)
# x_test_norm = scaler_x.transform(x_test)
# y_test_norm = scaler_y.transform(y_test)

# # print(x_train_norm[1:5])

# # 转为tensor并放到device
# x_train = torch.tensor(x_train_norm, dtype=torch.float64, device=device)
# y_train = torch.tensor(y_train_norm, dtype=torch.float64, device=device)
# x_test = torch.tensor(x_test_norm, dtype=torch.float64, device=device)
# y_test = torch.tensor(y_test_norm, dtype=torch.float64, device=device)

# # # 转为tensor并放到device
# # x_train = torch.tensor(x_train, dtype=torch.float64, device=device)
# # y_train = torch.tensor(y_train, dtype=torch.float64, device=device)
# # x_test = torch.tensor(x_test, dtype=torch.float64, device=device)
# # y_test = torch.tensor(y_test, dtype=torch.float64, device=device)


# print("训练集X形状:", x_train.shape)
# print("测试集X形状:", x_test.shape)
# # print("测试集", x_test,y_test)



**创建KAN模型**

In [3]:
kanmodel = MultKAN(width=[12,4,3,4], grid=5, k=3, seed=0, device=device)

checkpoint directory created: ./model
saving model version 0.0


创建数据集

In [4]:
from kan.utils import create_dataset_from_data

dataset = create_dataset_from_data(xdata, ydata, train_ratio=0.8, device=device)
dataset['train_input'].shape, dataset['train_label'].shape
dataset['test_input'].shape, dataset['test_label'].shape
print(dataset)

{'train_input': tensor([[1.1980e-01, 8.1948e-01, 5.1578e-02,  ..., 9.4613e-01, 6.9201e-01,
         2.4865e-01],
        [7.8886e-01, 1.2789e-01, 3.2867e-02,  ..., 9.1216e-02, 2.2765e-01,
         9.1423e-01],
        [5.2830e-01, 3.7701e-01, 9.8283e-02,  ..., 7.6724e-01, 6.5783e-01,
         1.1581e-01],
        ...,
        [4.1573e-01, 8.4354e-01, 1.1919e-01,  ..., 9.1365e-01, 8.2224e-01,
         3.9737e-02],
        [8.5421e-01, 5.3815e-01, 7.6739e-02,  ..., 9.5531e-01, 7.6862e-01,
         4.3488e-01],
        [1.4221e-01, 2.1496e-01, 6.7241e-04,  ..., 3.7173e-01, 8.2901e-02,
         3.7191e-01]]), 'test_input': tensor([[1.8266e-04, 4.2734e-01, 7.5860e-03,  ..., 8.5647e-01, 4.3775e-01,
         3.9155e-02],
        [1.1240e-01, 7.4382e-01, 6.6479e-03,  ..., 1.1295e-02, 1.1569e-02,
         5.6115e-01],
        [4.7518e-02, 5.3985e-01, 9.4723e-01,  ..., 6.4945e-01, 7.0387e-01,
         7.8791e-01],
        ...,
        [8.8871e-01, 7.8073e-01, 7.7304e-03,  ..., 1.6149e-01, 9.5064

训练模型

In [5]:
loss_result=kanmodel.fit(dataset, opt="Adam", steps=600, lamb=0.0001, lr=0.002);

| train_loss: 2.51e-02 | test_loss: 3.02e-02 | reg: 1.94e+01 | : 100%|█| 600/600 [00:21<00:00, 27.86


saving model version 0.1


预测结果输出

In [8]:
kanmodel.saveckpt('./kanmodel')

# newmodel=MultKAN(width=[11,5,4], grid=3, k=3, seed=0, device=device)
newmodel=MultKAN.loadckpt('./kanmodel')
# kanmodel.eval()
newmodel.eval()
# y_pred1= kanmodel(x_test)
y_pred2= newmodel(dataset['test_input'])
# print(y_pred1[10:25])
print(y_pred2[10:25])
print(dataset['test_label'][10:25])

tensor([[0.1441, 0.6015, 0.2426, 0.6862],
        [0.1659, 0.6824, 0.1761, 0.7607],
        [0.1902, 0.4908, 0.3442, 0.6012],
        [0.1003, 0.9521, 0.0226, 0.9601],
        [0.2560, 0.9105, 0.0388, 0.9399],
        [0.1214, 0.6045, 0.2348, 0.6863],
        [0.5919, 0.1572, 0.7355, 0.2471],
        [0.0533, 0.1140, 0.7578, 0.2151],
        [0.1703, 0.9560, 0.0193, 0.9700],
        [0.1487, 0.0807, 0.8276, 0.1854],
        [0.2571, 0.7559, 0.1323, 0.8267],
        [0.2356, 0.8131, 0.0960, 0.8682],
        [0.3419, 0.6340, 0.2178, 0.7235],
        [0.1363, 0.7907, 0.1108, 0.8471],
        [0.1822, 0.9268, 0.0334, 0.9494]], grad_fn=<SliceBackward0>)
tensor([[0.1399, 0.5933, 0.2564, 0.6631],
        [0.1742, 0.6824, 0.1726, 0.7516],
        [0.2078, 0.4688, 0.3449, 0.5460],
        [0.1286, 0.9822, 0.0126, 0.9854],
        [0.2832, 0.9145, 0.0444, 0.9444],
        [0.1016, 0.6074, 0.2449, 0.6969],
        [0.6661, 0.1509, 0.7473, 0.2070],
        [0.0297, 0.0763, 0.8567, 0.1752],
       